# KPI analysis & Performance Index

Core and advanced KPIs, **Performance Index Score** (weighted composite), and matplotlib charts. Figures can be saved to `images/` for README screenshots.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent if (Path.cwd().name == "notebooks") else Path.cwd()
IMG = ROOT / "images"
IMG.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(ROOT / "data" / "cleaned_data.csv", parse_dates=["Date"])
df["Year"] = df["Date"].dt.year
df["YearMonth"] = df["Date"].dt.to_period("M").dt.to_timestamp()
df.head()

## Core KPIs — portfolio totals
- Total Revenue  
- Profit Margin %  
- YoY / MoM Growth (on monthly series)  
- Average Order Value (Revenue / count of transactions)

## Advanced KPIs
- **CLV proxy**: average revenue per transaction × transactions per “customer” proxy (here: Region×Category segment frequency).  
- **Segment contribution %**  
- **Revenue concentration** (HHI-style on category shares)  
- **Profitability by Region**

In [ ]:
total_revenue = df["Revenue"].sum()
total_profit = df["Profit"].sum()
profit_margin_pct = (total_profit / total_revenue) * 100 if total_revenue else np.nan
aov = total_revenue / len(df)

monthly = df.groupby("YearMonth", as_index=False).agg(Revenue=("Revenue", "sum"))
monthly = monthly.sort_values("YearMonth")
monthly["MoM_growth_pct"] = monthly["Revenue"].pct_change() * 100
monthly["YoY_growth_pct"] = monthly["Revenue"].pct_change(periods=12) * 100

latest_mom = monthly["MoM_growth_pct"].iloc[-1]
latest_yoy = monthly["YoY_growth_pct"].iloc[-1]

seg = df.groupby(["Region", "Category"], as_index=False).agg(
    Revenue=("Revenue", "sum"), Orders=("Revenue", "count")
)
seg["Avg_Order_Value"] = seg["Revenue"] / seg["Orders"]
seg["CLV_proxy"] = seg["Avg_Order_Value"] * seg["Orders"]

cat_share = df.groupby("Category")["Revenue"].sum()
contrib = (cat_share / cat_share.sum() * 100).sort_values(ascending=False)
hhi = float(((cat_share / cat_share.sum()) ** 2).sum())

region_prof = df.groupby("Region", as_index=False).agg(
    Revenue=("Revenue", "sum"), Profit=("Profit", "sum")
)
region_prof["Profit_Margin_Pct"] = (region_prof["Profit"] / region_prof["Revenue"]) * 100

print(f"Total Revenue: {total_revenue:,.2f}")
print(f"Profit Margin %: {profit_margin_pct:.2f}")
print(f"Average order value (avg revenue per row): {aov:,.2f}")
print(f"Latest MoM growth %: {latest_mom:.2f}")
print(f"Latest YoY growth % (if 12+ months): {latest_yoy}")
print(f"Revenue concentration index (HHI on categories, 0-1): {hhi:.3f}")
display(contrib.head(10).to_frame("Contribution_pct"))
display(region_prof.sort_values("Revenue", ascending=False))

## Performance Index Score (weighted KPI)

```
Performance Score = 
  0.4 * (normalized revenue growth) +
  0.3 * (normalized profit margin) +
  0.3 * (normalized sales volume)
```

Computed **by Region** using YoY revenue growth (last vs prior year), average profit margin, and total quantity sold. Each component is min–max normalized across regions.

In [ ]:
def min_max(s: pd.Series) -> pd.Series:
    lo, hi = s.min(), s.max()
    if hi == lo:
        return pd.Series(1.0, index=s.index)
    return (s - lo) / (hi - lo)


rev_y = df.groupby(["Region", "Year"])["Revenue"].sum().unstack("Year")
years_sorted = sorted(rev_y.columns)
if len(years_sorted) >= 2:
    y1, y0 = years_sorted[-1], years_sorted[-2]
    rev_growth = ((rev_y[y1] - rev_y[y0]) / rev_y[y0].replace(0, np.nan)).fillna(0) * 100
else:
    rev_growth = pd.Series(0.0, index=rev_y.index)

rp_reg = df.groupby("Region", as_index=False).agg(
    Revenue=("Revenue", "sum"), Profit=("Profit", "sum")
)
rp_reg["Profit_Margin_pct"] = (rp_reg["Profit"] / rp_reg["Revenue"]) * 100
rm = rp_reg.set_index("Region")["Profit_Margin_pct"]
vol = df.groupby("Region")["Quantity"].sum()

score_df = (
    pd.DataFrame(
        {"Revenue_Growth_pct": rev_growth, "Profit_Margin_pct": rm, "Sales_Volume": vol}
    )
    .fillna(0)
    .sort_index()
)

score_df["n_growth"] = min_max(score_df["Revenue_Growth_pct"])
score_df["n_margin"] = min_max(score_df["Profit_Margin_pct"])
score_df["n_volume"] = min_max(score_df["Sales_Volume"])
score_df["Performance_Score"] = (
    0.4 * score_df["n_growth"] + 0.3 * score_df["n_margin"] + 0.3 * score_df["n_volume"]
)

top5 = score_df.sort_values("Performance_Score", ascending=False).head(5)
worst_seg = (
    df.groupby("Category", as_index=False)
    .agg(Revenue=("Revenue", "sum"), Profit=("Profit", "sum"))
    .assign(Profit_Margin_pct=lambda x: (x["Profit"] / x["Revenue"]) * 100)
    .sort_values("Profit_Margin_pct")
    .head(5)
)

print("Top 5 regions by Performance Score:")
display(top5[["Performance_Score", "Revenue_Growth_pct", "Profit_Margin_pct", "Sales_Volume"]])
print("Worst-performing segments by profit margin:")
display(worst_seg)

## Visualizations (matplotlib)
Save copies to `images/` for GitHub / README.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
m = df.groupby("YearMonth", as_index=False)["Revenue"].sum()
ax.plot(m["YearMonth"], m["Revenue"], color="#1f77b4", lw=2)
ax.set_title("Revenue trend")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue")
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(IMG / "kpi_trends.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
t = top5.sort_values("Performance_Score", ascending=True)
ax.barh(range(len(t)), t["Performance_Score"], color="#ff7f0e")
ax.set_yticks(range(len(t)))
ax.set_yticklabels(list(t.index))
ax.set_xlabel("Performance Score")
ax.set_title("Top 5 regions — Performance Index")
plt.tight_layout()
fig.savefig(IMG / "performance_top5_regions.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
rp = region_prof.sort_values("Revenue", ascending=False)
ax.bar(rp["Region"], rp["Profit_Margin_pct"], color="#2ca02c")
ax.set_title("Profitability by region (margin %)")
ax.set_ylabel("Profit margin %")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
fig.savefig(IMG / "profitability_by_region.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(6, 3))
kpi_vals = [total_revenue / 1e6, profit_margin_pct, aov / 1e3]
kpi_lbls = ["Revenue (M)", "Margin %", "AOV (k)"]
ax.bar(kpi_lbls, kpi_vals, color=["#1f77b4", "#9467bd", "#8c564b"])
ax.set_title("KPI snapshot (scaled units for chart)")
plt.tight_layout()
fig.savefig(IMG / "kpi_summary_bars.png", dpi=150)
plt.show()

print("Saved figures under:", IMG)

### Optional composite “dashboard preview”
Combine a few subplots into one image for `images/dashboard_preview.png`.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
m = df.groupby("YearMonth", as_index=False)["Revenue"].sum()
axes[0, 0].plot(m["YearMonth"], m["Revenue"], color="#1f77b4")
axes[0, 0].set_title("Revenue trend")
axes[0, 0].grid(True, alpha=0.3)

cr = df.groupby("Category")["Revenue"].sum().sort_values(ascending=False)
axes[0, 1].bar(cr.index, cr.values, color="#2ca02c")
axes[0, 1].set_title("Revenue by category")
axes[0, 1].tick_params(axis="x", rotation=35)

_t = top5.sort_values("Performance_Score", ascending=True)
axes[1, 0].barh(range(len(_t)), _t["Performance_Score"], color="#ff7f0e")
axes[1, 0].set_yticks(range(len(_t)))
axes[1, 0].set_yticklabels(list(_t.index))
axes[1, 0].set_title("Top regions — performance score")

rp = region_prof.sort_values("Revenue", ascending=False)
axes[1, 1].bar(rp["Region"], rp["Profit_Margin_pct"], color="#d62728")
axes[1, 1].set_title("Margin % by region")
axes[1, 1].tick_params(axis="x", rotation=25)

plt.tight_layout()
fig.savefig(IMG / "dashboard_preview.png", dpi=150)
plt.show()